# Fuzzy Test Data Setup

Voegt **bewust inconsistente** entiteiten toe aan Salesforce, SmartSales en de inbox.
De namen komen **niet overeen** tussen systemen — dit is de use case die graph RAG oplost.

| Echte entiteit | SF naam | SS locatienaam | Email domein | Waarom moeilijk |
|---|---|---|---|---|
| Carrefour | `Carrefour Belgium SA` | `CRF - Anderlecht` | `ops@carrefour-be.eu` | Afkorting + andere TLD |
| Delhaize | `Delhaize Group` | `AD Delhaize - Forest` | `procurement@delhaize.eu` | Franchise prefix, `.eu` |
| AB InBev | `Anheuser-Busch InBev` | `AB InBev - Leuven` | `sales@ab-inbev.com` | Volledige juridische naam vs afkorting |
| Proximus | *(geen SF account)* | *(geen SS locatie)* | `info@proximus.be` | Bestaat alleen in email — gap detection |
| Lidl | *(geen SF account)* | `Lidl Belgium - Antwerp` | *(geen email)* | Bestaat alleen in SS — gap detection |

**Doel:** prompts zoals *"Which email senders have an active deal?"* falen op het huidige systeem
omdat `carrefour-be.eu` ≠ `Carrefour Belgium SA` zonder entity resolution.

## 0. Setup & Auth

In [3]:
import sys, os
# Kernel working dir is eval/testdata/ — zet naar project root zodat relatieve
# paden (salesforce.key, config.cfg, .env) correct werken.
os.chdir(os.path.abspath('../..'))
sys.path.insert(0, os.getcwd())

import httpx, json as _json
from dotenv import load_dotenv
load_dotenv()

# ── Salesforce ──────────────────────────────────────────────────────────────
from salesforce.auth import authenticate_salesforce
sf_creds = authenticate_salesforce('https://test.salesforce.com')
SF_TOKEN = sf_creds.access_token
SF_BASE  = sf_creds.instance_url.rstrip('/')
SF_API   = f'{SF_BASE}/services/data/v59.0'
print(f'✓ Salesforce  instance={SF_BASE}')

# ── SmartSales ──────────────────────────────────────────────────────────────
from smartsales.auth import authenticate_from_env
ss_creds = authenticate_from_env()
SS_TOKEN = ss_creds.access_token
SS_BASE  = 'https://proxy-smartsales.easi.net/proxy/rest'
print(f'✓ SmartSales  token_ok=True')

✓ Salesforce  instance=https://demoeasi2023--aiseach.sandbox.my.salesforce.com
✓ SmartSales  token_ok=True


In [5]:
# ── Helpers (identiek aan testdata_setup.ipynb) ──────────────────────────────

def sf_post(obj_type: str, payload: dict) -> dict:
    url = f'{SF_API}/sobjects/{obj_type}/'
    r = httpx.post(url, json=payload, headers={'Authorization': f'Bearer {SF_TOKEN}'})
    if r.status_code == 400:
        try:
            for err in r.json():
                if err.get('errorCode') == 'DUPLICATES_DETECTED':
                    match_records = (err.get('duplicateResult', {})
                                       .get('matchResults', [{}])[0]
                                       .get('matchRecords', [{}]))
                    if match_records:
                        eid = match_records[0]['record']['Id']
                        print(f'↩ {obj_type} bestaat al — hergebruik  id={eid}')
                        return {'id': eid, 'success': True, 'reused': True}
        except Exception:
            pass
        print(f'❌ Salesforce fout [{r.status_code}]: {r.text}')
        r.raise_for_status()
    r.raise_for_status()
    return r.json()

def sf_query(soql: str) -> list:
    r = httpx.get(f'{SF_API}/query', params={'q': soql},
                  headers={'Authorization': f'Bearer {SF_TOKEN}'})
    r.raise_for_status()
    return r.json().get('records', [])

def sf_get_account_id(name: str) -> str | None:
    recs = sf_query(f"SELECT Id FROM Account WHERE Name = '{name}' LIMIT 1")
    return recs[0]['Id'] if recs else None

def ss_get_or_create(payload: dict) -> dict:
    name = payload['name']
    q = _json.dumps({'name': f'eq:{name}'})
    r = httpx.get(f'{SS_BASE}/api/v3/location/list',
                  params={'q': q, 'p': 'simple'},
                  headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = r.json().get('entries', [])
    if entries:
        loc = entries[0]
        print(f'↩ Locatie "{name}" bestaat al  uid={loc.get("uid")}')
        return loc
    r2 = httpx.post(f'{SS_BASE}/api/v3/location', json=payload,
                    headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r2.is_success:
        print(f'❌ SmartSales fout [{r2.status_code}]: {r2.text}')
        r2.raise_for_status()
    result = r2.json()
    print(f'✓ Locatie "{name}" aangemaakt  uid={result.get("uid")}')
    return result

def ss_get_or_create_catalog_item(payload: dict) -> dict:
    code = payload['code']
    r = httpx.get(f'{SS_BASE}/api/v3/catalog/list',
                  params={'p': 'simple'},
                  headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = [e for e in r.json().get('entries', []) if e.get('code') == code]
    if entries:
        item = entries[0]
        print(f'↩ Catalog item "{item.get("title") or code}" bestaat al  uid={item.get("uid")}')
        return item
    r2 = httpx.post(f'{SS_BASE}/api/v3/catalog/item', json=payload,
                    headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text}')
        r2.raise_for_status()
    item = r2.json()
    print(f'✓ Catalog item "{item.get("title") or code}" aangemaakt  uid={item.get("uid")}')
    return item

def ss_get_location(name: str) -> dict | None:
    q = _json.dumps({'name': f'eq:{name}'})
    r = httpx.get(f'{SS_BASE}/api/v3/location/list',
                  params={'q': q, 'p': 'simple'},
                  headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = r.json().get('entries', [])
    if not entries:
        print(f'⚠️  Locatie "{name}" niet gevonden')
        return None
    loc = entries[0]
    return {'uid': loc['uid'], 'code': loc.get('code', ''), 'name': loc.get('name', ''), 'externalId': None}

def ss_get_or_create_order(ref: str, supplier: dict, customer: dict, items: list, comments: str = '') -> dict:
    from datetime import datetime, timezone
    q = _json.dumps({'internalReference': f'eq:{ref}'})
    r = httpx.get(f'{SS_BASE}/api/v3/order/list',
                  params={'q': q, 'p': 'simple'},
                  headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if r.is_success:
        entries = r.json().get('entries', [])
        if entries:
            print(f'↩ Order "{ref}" bestaat al  uid={entries[0].get("uid")}')
            return entries[0]
    date_str = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S000+0000')
    order_items, subtotal = [], 0.0
    for it in items:
        total = it['price'] * it['qty']
        subtotal += total
        order_items.append({
            'code': it['code'], 'description': it['title'],
            'quantity': it['qty'], 'price': it['price'],
            'totalPrice': total, 'finalDiscountPrice': total,
            'discount': 0, 'discountIsPercentage': False, 'free': False,
            'hasAutoDiscounts': False, 'autoDiscounts': [],
            'hasOverriddenAutoDiscount': False, 'priceManuallySet': False,
        })
    payload = {
        'date': date_str, 'internalReference': ref, 'externalReference': ref,
        'type': 'ORDER', 'approbationStatus': 'APPROVED', 'comments': comments,
        'locale': 'en', 'subtotal': subtotal, 'total': subtotal,
        'totalQuantity': sum(it['qty'] for it in items),
        'supplier': supplier, 'customer': customer, 'items': order_items,
        'hasAutoDiscounts': False, 'autoDiscounts': [], 'totalModifiers': [],
        'offer': False, 'discountVisible': False, 'form': {}, 'customFields': {},
    }
    r2 = httpx.post(f'{SS_BASE}/api/v3/order', json=payload,
                    headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text}')
        r2.raise_for_status()
    order = r2.json()
    print(f'✓ Order "{ref}"  uid={order.get("uid")}  supplier={supplier["name"]}')
    return order

GRP_HARDWARE = {'uid': 'a708e627-38ac-11f1-a803-005056010707', 'code': 'GRP-IT-HARDWARE', 'title': 'IT Hardware'}
GRP_FOOD     = {'uid': 'e900d857-2856-11f1-a2fe-005056010707', 'code': 'SPREADS', 'title': 'Spreads'}

print('✓ helpers geladen')

✓ helpers geladen


---
## 1. Salesforce — Fuzzy accounts

Formele/juridische namen die **niet overeenkomen** met SS-locatienamen of email-domeinen.

In [6]:
# Carrefour: formele juridische naam in SF
existing = sf_get_account_id('Carrefour Belgium SA')
if existing:
    CARREFOUR_ID = existing
    print(f'Account Carrefour Belgium SA bestaat al  id={CARREFOUR_ID}')
else:
    res = sf_post('Account', {
        'Name': 'Carrefour Belgium SA',          # ← juridische naam, NIET 'Carrefour' of 'CRF'
        'Industry': 'Retail',
        'BillingCity': 'Evere',
        'BillingCountry': 'Belgium',
        'Website': 'https://www.carrefour.be',
        'Description': 'Belgian subsidiary of Carrefour Group. Operates hypermarkets and supermarkets.',
    })
    CARREFOUR_ID = res['id']
    print(f'Account Carrefour Belgium SA aangemaakt  id={CARREFOUR_ID}')

res = sf_post('Opportunity', {
    'Name': 'Carrefour Belgium SA — In-Store Analytics Pilot',
    'StageName': 'Proposal/Price Quote',
    'CloseDate': '2026-07-31',
    'Amount': 175000,
    'AccountId': CARREFOUR_ID,
    'Description': 'Pilot for in-store analytics across 12 Belgian hypermarkets.',
})
print(f'Opportunity Carrefour  id={res["id"]}')

res = sf_post('Case', {
    'Subject': 'Carrefour Belgium SA — POS integration delay',
    'Status': 'New',
    'Priority': 'High',
    'AccountId': CARREFOUR_ID,
})
print(f'Case Carrefour  id={res["id"]}')

Account Carrefour Belgium SA aangemaakt  id=001KI00000N80nMYAR
Opportunity Carrefour  id=006KI000005bIGDYA2
Case Carrefour  id=500KI000001mK0yYAE


In [7]:
# Delhaize: groepsnaam in SF, franchisenaam in SS
existing = sf_get_account_id('Delhaize Group')
if existing:
    DELHAIZE_ID = existing
    print(f'Account Delhaize Group bestaat al  id={DELHAIZE_ID}')
else:
    res = sf_post('Account', {
        'Name': 'Delhaize Group',                # ← groepsnaam, SS gebruikt 'AD Delhaize - Forest'
        'Industry': 'Retail',
        'BillingCity': 'Brussels',
        'BillingCountry': 'Belgium',
        'Website': 'https://www.delhaize.be',
        'Description': 'Belgian supermarket group. Franchise stores operate under AD Delhaize brand.',
    })
    DELHAIZE_ID = res['id']
    print(f'Account Delhaize Group aangemaakt  id={DELHAIZE_ID}')

res = sf_post('Opportunity', {
    'Name': 'Delhaize Group — Supply Chain Optimisation',
    'StageName': 'Negotiation/Review',
    'CloseDate': '2026-08-31',
    'Amount': 310000,
    'AccountId': DELHAIZE_ID,
})
print(f'Opportunity Delhaize  id={res["id"]}')

res = sf_post('Case', {
    'Subject': 'Delhaize Group — EDI connection issue',
    'Status': 'Working',
    'Priority': 'Medium',
    'AccountId': DELHAIZE_ID,
})
print(f'Case Delhaize  id={res["id"]}')

Account Delhaize Group aangemaakt  id=001KI00000N7zVuYAJ
Opportunity Delhaize  id=006KI000005bIGIYA2
Case Delhaize  id=500KI000001mK13YAE


In [8]:
# AB InBev: volledige juridische naam in SF, afkorting in SS
existing = sf_get_account_id('Anheuser-Busch InBev')
if existing:
    ABINBEV_ID = existing
    print(f'Account Anheuser-Busch InBev bestaat al  id={ABINBEV_ID}')
else:
    res = sf_post('Account', {
        'Name': 'Anheuser-Busch InBev',          # ← juridisch, SS en emails gebruiken 'AB InBev'
        'Industry': 'Food & Beverage',
        'BillingCity': 'Leuven',
        'BillingCountry': 'Belgium',
        'Website': 'https://www.ab-inbev.com',
        'Description': 'World largest brewer. Belgian HQ in Leuven. Known as AB InBev.',
    })
    ABINBEV_ID = res['id']
    print(f'Account Anheuser-Busch InBev aangemaakt  id={ABINBEV_ID}')

res = sf_post('Opportunity', {
    'Name': 'Anheuser-Busch InBev — Distribution Network Mapping',
    'StageName': 'Qualification',
    'CloseDate': '2026-09-15',
    'Amount': 520000,
    'AccountId': ABINBEV_ID,
})
print(f'Opportunity AB InBev  id={res["id"]}')

Account Anheuser-Busch InBev aangemaakt  id=001KI00000N80nRYAR
Opportunity AB InBev  id=006KI000005bIGNYA2


---
## 2. SmartSales — Fuzzy locaties

Namen die **niet overeenkomen** met de SF-accountnamen hierboven.

In [9]:
# Carrefour in SS: afkorting 'CRF' + stad — niet 'Carrefour Belgium SA'
LOC_CRF = ss_get_or_create({
    'name': 'CRF - Anderlecht',           # ← 'CRF' is niet 'Carrefour Belgium SA'
    'code': 'CRF-AND-001',
    'city': 'Brussels',
    'country': 'Belgium',
    'street': 'Bergensesteenweg 1424',
    'zip': '1070',
})

# Delhaize in SS: franchise prefix 'AD' + stad — niet 'Delhaize Group'
LOC_ADL = ss_get_or_create({
    'name': 'AD Delhaize - Forest',       # ← 'AD Delhaize' franchise, niet 'Delhaize Group'
    'code': 'ADL-FOR-001',
    'city': 'Brussels',
    'country': 'Belgium',
    'street': 'Chaussée de Neerstalle 800',
    'zip': '1190',
})

# AB InBev in SS: afkorting + stad — niet 'Anheuser-Busch InBev'
LOC_ABI = ss_get_or_create({
    'name': 'AB InBev - Leuven',          # ← 'AB InBev' is niet 'Anheuser-Busch InBev'
    'code': 'ABI-LEU-001',
    'city': 'Leuven',
    'country': 'Belgium',
    'street': 'Vaartstraat 94',
    'zip': '3000',
})

# Lidl: ALLEEN in SS, geen SF account, geen emails
LOC_LIDL = ss_get_or_create({
    'name': 'Lidl Belgium - Antwerp',     # ← bestaat niet in SF of inbox
    'code': 'LDL-ANT-001',
    'city': 'Antwerp',
    'country': 'Belgium',
    'street': 'Noordersingel 10',
    'zip': '2140',
})

print()
for label, loc in [('CRF - Anderlecht', LOC_CRF), ('AD Delhaize - Forest', LOC_ADL),
                    ('AB InBev - Leuven', LOC_ABI), ('Lidl Belgium - Antwerp', LOC_LIDL)]:
    if loc:
        print(f'  {label:<30}  uid={loc.get("uid")}')

✓ Locatie "CRF - Anderlecht" aangemaakt  uid=2d79461f-4310-11f1-a803-005056010707
✓ Locatie "AD Delhaize - Forest" aangemaakt  uid=2e532129-4310-11f1-a803-005056010707
✓ Locatie "AB InBev - Leuven" aangemaakt  uid=2f355296-4310-11f1-a803-005056010707
✓ Locatie "Lidl Belgium - Antwerp" aangemaakt  uid=301c94e5-4310-11f1-a803-005056010707

  CRF - Anderlecht                uid=2d79461f-4310-11f1-a803-005056010707
  AD Delhaize - Forest            uid=2e532129-4310-11f1-a803-005056010707
  AB InBev - Leuven               uid=2f355296-4310-11f1-a803-005056010707
  Lidl Belgium - Antwerp          uid=301c94e5-4310-11f1-a803-005056010707


---
## 3. SmartSales — Catalog items & Orders voor fuzzy locaties

In [10]:
# Catalog items
CAT_CRF = ss_get_or_create_catalog_item({
    'code': 'PROD-CRF-001',
    'title': 'Carrefour — In-Store Analytics Module',
    'description': 'Analytics module for Carrefour hypermarket locations.',
    'price': 4500.00, 'salesUnit': 1, 'unitOfMeasure': 'license',
    'group': GRP_HARDWARE,
})

CAT_ADL = ss_get_or_create_catalog_item({
    'code': 'PROD-ADL-001',
    'title': 'Delhaize — EDI Connector Pack',
    'description': 'EDI integration package for Delhaize franchise stores.',
    'price': 1200.00, 'salesUnit': 1, 'unitOfMeasure': 'piece',
    'group': GRP_HARDWARE,
})

CAT_ABI = ss_get_or_create_catalog_item({
    'code': 'PROD-ABI-001',
    'title': 'AB InBev — Distribution Mapping License',
    'description': 'Annual license for distribution network mapping tool. AB InBev Leuven HQ.',
    'price': 8750.00, 'salesUnit': 1, 'unitOfMeasure': 'license',
    'group': GRP_HARDWARE,
})

CAT_LIDL = ss_get_or_create_catalog_item({
    'code': 'PROD-LDL-001',
    'title': 'Lidl — Shelf Replenishment Kit',
    'description': 'Standard shelf replenishment package for Lidl Belgium stores.',
    'price': 299.00, 'salesUnit': 5, 'unitOfMeasure': 'piece',
    'group': GRP_HARDWARE,
})

✓ Catalog item "Carrefour — In-Store Analytics Module" aangemaakt  uid=31d61ac2-4310-11f1-a803-005056010707
✓ Catalog item "Delhaize — EDI Connector Pack" aangemaakt  uid=32a9422b-4310-11f1-a803-005056010707
✓ Catalog item "AB InBev — Distribution Mapping License" aangemaakt  uid=33676a4f-4310-11f1-a803-005056010707
✓ Catalog item "Lidl — Shelf Replenishment Kit" aangemaakt  uid=34579c1a-4310-11f1-a803-005056010707


In [11]:
# Customer1 locatie ophalen (bestaande testlocatie)
LOC_CUSTOMER1 = ss_get_location('Customer1')
if not LOC_CUSTOMER1:
    print('⚠️  Customer1 niet gevonden — run eerst testdata_setup.ipynb')

# Orders aanmaken
if LOC_CRF and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-CRF-2026-001', supplier=LOC_CRF, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-CRF-001', 'title': 'Carrefour — In-Store Analytics Module',
                'qty': 12, 'price': 4500.00}],
        comments='Analytics pilot across 12 Carrefour BE hypermarkets. Ref: Carrefour Belgium SA opp.',
    )

if LOC_ADL and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-ADL-2026-001', supplier=LOC_ADL, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-ADL-001', 'title': 'Delhaize — EDI Connector Pack',
                'qty': 8, 'price': 1200.00}],
        comments='EDI connectors for AD Delhaize franchise rollout. Parent account: Delhaize Group.',
    )

if LOC_ABI and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-ABI-2026-001', supplier=LOC_ABI, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-ABI-001', 'title': 'AB InBev — Distribution Mapping License',
                'qty': 1, 'price': 8750.00}],
        comments='Distribution mapping for Anheuser-Busch InBev network. Largest order this quarter.',
    )

if LOC_LIDL and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-LDL-2026-001', supplier=LOC_LIDL, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-LDL-001', 'title': 'Lidl — Shelf Replenishment Kit',
                'qty': 20, 'price': 299.00}],
        comments='Shelf replenishment stock for Lidl Belgium Antwerp. No SF account linked.',
    )

✓ Order "ORD-CRF-2026-001"  uid=36cc5831-4310-11f1-a803-005056010707  supplier=CRF - Anderlecht
✓ Order "ORD-ADL-2026-001"  uid=37bc49f5-4310-11f1-a803-005056010707  supplier=AD Delhaize - Forest
✓ Order "ORD-ABI-2026-001"  uid=38ad6cf1-4310-11f1-a803-005056010707  supplier=AB InBev - Leuven
✓ Order "ORD-LDL-2026-001"  uid=39821838-4310-11f1-a803-005056010707  supplier=Lidl Belgium - Antwerp


---
## 4. Microsoft 365 — Fuzzy inbox emails

Afzenderdomeinen komen **niet overeen** met SF-accountnamen:
- `carrefour-be.eu` ≠ `Carrefour Belgium SA`
- `delhaize.eu` ≠ `Delhaize Group`  
- `ab-inbev.com` ≠ `Anheuser-Busch InBev`
- `proximus.be` → **geen SF account, geen SS locatie** (gap detection)

In [18]:
import requests as _req

MAILBOX   = 'smartadmin@easigroupdemo.onmicrosoft.com'
TENANT_ID = os.getenv('test_email_injector_tenant_id')
CLIENT_ID = os.getenv('test_email_injector_client_id')
CLIENT_SECRET = os.getenv('test_email_injector_secret_value')

def _get_graph_token():
    r = _req.post(
        f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
        data={
            'grant_type': 'client_credentials',
            'client_id': CLIENT_ID,
            'client_secret': CLIENT_SECRET,
            'scope': 'https://graph.microsoft.com/.default',
        },
    )
    r.raise_for_status()
    return r.json()['access_token']

GRAPH_TOKEN = _get_graph_token()
GRAPH_BASE  = 'https://graph.microsoft.com/v1.0'
print(f'✓ Graph auth  mailbox={MAILBOX}')

✓ Graph auth  mailbox=smartadmin@easigroupdemo.onmicrosoft.com


In [19]:
def graph_create_inbox_message_idempotent(subject: str, body: str,
                                           from_address: str, from_name: str) -> None:
    headers = {'Authorization': f'Bearer {GRAPH_TOKEN}', 'Content-Type': 'application/json'}

    # Idempotency check
    r = _req.get(
        f"{GRAPH_BASE}/users/{MAILBOX}/messages",
        params={'$filter': f"subject eq '{subject}'", '$top': '1', '$select': 'id'},
        headers=headers,
    )
    if r.ok and r.json().get('value'):
        print(f'↩ Inbox email al aanwezig: "{subject}"')
        return

    # Aanmaken
    payload = {
        'subject': subject,
        'body': {'contentType': 'Text', 'content': body},
        'from':         {'emailAddress': {'address': from_address, 'name': from_name}},
        'sender':       {'emailAddress': {'address': from_address, 'name': from_name}},
        'toRecipients': [{'emailAddress': {'address': MAILBOX}}],
    }
    r2 = _req.post(f"{GRAPH_BASE}/users/{MAILBOX}/messages", json=payload, headers=headers)
    if not r2.ok:
        print(f'❌ [{r2.status_code}]: {r2.text[:200]}')
        r2.raise_for_status()
    msg_id = r2.json()['id']

    # Verplaats naar inbox
    _req.post(
        f"{GRAPH_BASE}/users/{MAILBOX}/messages/{msg_id}/move",
        json={'destinationId': 'inbox'},
        headers=headers,
    ).raise_for_status()

    print(f'✓ Inbox email: "{subject}"  (van: {from_name} <{from_address}>)')

print('✓ graph helper geladen')

✓ graph helper geladen


In [20]:
# Carrefour: domein 'carrefour-be.eu' ≠ SF naam 'Carrefour Belgium SA'
graph_create_inbox_message_idempotent(
    subject='Q2 pilot update — analytics rollout',
    body="""Hi,

Quick update on the in-store analytics pilot we discussed.
Our team at the Anderlecht hypermarket has completed the initial installation.
We would like to schedule a review for the 12 pilot locations.

Please confirm your availability for the week of May 20.

Kind regards,
Operations Team
Carrefour Belgium — Anderlecht""",
    from_address='ops@carrefour-be.eu',   # ← carrefour-be.eu ≠ 'Carrefour Belgium SA'
    from_name='Carrefour BE Operations',
)

graph_create_inbox_message_idempotent(
    subject='Offerte aanvraag — POS koppeling modules',
    body="""Goedemiddag,

Wij zijn op zoek naar een oplossing voor de POS-integratie in onze Belgische vestigingen.
Carrefour Belgium SA heeft momenteel 47 hypermarkten in België.
Graag ontvangen wij een offerte voor de analytics module.

Met vriendelijke groeten,
Procurement — CRF Belgium""",
    from_address='procurement@carrefour-be.eu',
    from_name='CRF Belgium Procurement',
)

# Delhaize: domein 'delhaize.eu' ≠ SF naam 'Delhaize Group'
graph_create_inbox_message_idempotent(
    subject='EDI connector — technische vraag',
    body="""Dag,

We hebben een technisch probleem met de EDI-koppeling in onze Forest-winkel (AD Delhaize).
Kunnen jullie iemand sturen voor een check? Het gaat over de bestelling ORD-ADL-2026-001.

Groeten,
IT Support — AD Delhaize Forest""",
    from_address='it@delhaize.eu',        # ← delhaize.eu ≠ 'Delhaize Group'
    from_name='AD Delhaize IT',
)

graph_create_inbox_message_idempotent(
    subject='Supply chain project — follow-up meeting',
    body="""Hello,

Following up on the supply chain optimisation project for Delhaize.
Our procurement team (procurement@delhaize.eu) would like to review the proposal.
The project scope covers all AD Delhaize franchise locations in Belgium.

Best,
Delhaize Group — Project Office""",
    from_address='procurement@delhaize.eu',
    from_name='Delhaize Procurement',
)

# AB InBev: domein 'ab-inbev.com' ≠ SF naam 'Anheuser-Busch InBev'
graph_create_inbox_message_idempotent(
    subject='Distribution mapping — scope confirmation',
    body="""Hi,

Confirming the scope for the AB InBev distribution mapping project.
The license should cover our full Belgian distribution network (Leuven HQ + regional depots).
Contract reference: Anheuser-Busch InBev — Distribution Network Mapping.

Regards,
AB InBev — Sales Operations
Vaartstraat 94, Leuven""",
    from_address='sales@ab-inbev.com',    # ← ab-inbev.com ≠ 'Anheuser-Busch InBev'
    from_name='AB InBev Sales',
)

# Proximus: ALLEEN in email — geen SF account, geen SS locatie
graph_create_inbox_message_idempotent(
    subject='Proximus — partnership inquiry',
    body="""Hello,

I'm reaching out from Proximus Business Solutions.
We are exploring potential partnerships for field sales tooling.
Would you be available for a short call next week?

Kind regards,
Business Development — Proximus""",
    from_address='info@proximus.be',      # ← geen SF account, geen SS locatie
    from_name='Proximus Business',
)

✓ Inbox email: "Q2 pilot update — analytics rollout"  (van: Carrefour BE Operations <ops@carrefour-be.eu>)
✓ Inbox email: "Offerte aanvraag — POS koppeling modules"  (van: CRF Belgium Procurement <procurement@carrefour-be.eu>)
✓ Inbox email: "EDI connector — technische vraag"  (van: AD Delhaize IT <it@delhaize.eu>)
✓ Inbox email: "Supply chain project — follow-up meeting"  (van: Delhaize Procurement <procurement@delhaize.eu>)
✓ Inbox email: "Distribution mapping — scope confirmation"  (van: AB InBev Sales <sales@ab-inbev.com>)
✓ Inbox email: "Proximus — partnership inquiry"  (van: Proximus Business <info@proximus.be>)


---
## 5. Verificatie

Overzicht van wat aangemaakt is en wat de verwachte entity resolution uitdagingen zijn.

In [21]:
# Salesforce overzicht
fuzzy_names = ['Carrefour Belgium SA', 'Delhaize Group', 'Anheuser-Busch InBev']
accounts = [r for name in fuzzy_names
            for r in sf_query(f"SELECT Id, Name, BillingCity FROM Account WHERE Name = '{name}' LIMIT 1")]

print('=== Salesforce fuzzy accounts ===')
for a in accounts:
    print(f"  SF: {a['Name']:<30} [{a['Id']}]")

print()
print('=== SmartSales fuzzy locaties ===')
for label, loc in [('CRF - Anderlecht', LOC_CRF), ('AD Delhaize - Forest', LOC_ADL),
                    ('AB InBev - Leuven', LOC_ABI), ('Lidl Belgium - Antwerp', LOC_LIDL)]:
    uid = loc.get('uid') if loc else '?'
    print(f"  SS: {label:<30} uid={uid}")

print()
print('=== Inbox emails (fuzzy domeinen) ===')
fuzzy_domains = [
    ('ops@carrefour-be.eu',       'Carrefour Belgium SA', 'CRF - Anderlecht'),
    ('procurement@delhaize.eu',   'Delhaize Group',       'AD Delhaize - Forest'),
    ('sales@ab-inbev.com',        'Anheuser-Busch InBev', 'AB InBev - Leuven'),
    ('info@proximus.be',          '(geen SF account)',    '(geen SS locatie)'),
]
for domain, sf_name, ss_name in fuzzy_domains:
    print(f"  Email: {domain:<35} → SF: {sf_name:<25} → SS: {ss_name}")

print()
print('=== Gap detection cases ===')
print('  Lidl Belgium - Antwerp  → SS only, geen SF account, geen email')
print('  Proximus                → email only, geen SF, geen SS')

=== Salesforce fuzzy accounts ===
  SF: Carrefour Belgium SA           [001KI00000N80nMYAR]
  SF: Delhaize Group                 [001KI00000N7zVuYAJ]
  SF: Anheuser-Busch InBev           [001KI00000N80nRYAR]

=== SmartSales fuzzy locaties ===
  SS: CRF - Anderlecht               uid=2d79461f-4310-11f1-a803-005056010707
  SS: AD Delhaize - Forest           uid=2e532129-4310-11f1-a803-005056010707
  SS: AB InBev - Leuven              uid=2f355296-4310-11f1-a803-005056010707
  SS: Lidl Belgium - Antwerp         uid=301c94e5-4310-11f1-a803-005056010707

=== Inbox emails (fuzzy domeinen) ===
  Email: ops@carrefour-be.eu                 → SF: Carrefour Belgium SA      → SS: CRF - Anderlecht
  Email: procurement@delhaize.eu             → SF: Delhaize Group            → SS: AD Delhaize - Forest
  Email: sales@ab-inbev.com                  → SF: Anheuser-Busch InBev      → SS: AB InBev - Leuven
  Email: info@proximus.be                    → SF: (geen SF account)         → SS: (geen SS locatie)


---
## Wat dit aantoont

| Prompt | Huidig systeem (zonder graph RAG) | Met graph RAG |
|--------|-----------------------------------|---------------|
| *"Which email senders have an active deal?"* | Vindt `carrefour-be.eu` niet terug als `Carrefour Belgium SA` → **mist match** | Graph legt link `carrefour-be.eu → Carrefour Belgium SA` → **correct** |
| *"Which companies emailed me but have no SS location?"* | Kan `ops@carrefour-be.eu` niet matchen aan `CRF - Anderlecht` → **fout resultaat** | Graph weet dat `carrefour-be.eu` = `CRF - Anderlecht` → **correct** |
| *"Which companies have an open order but no SF account?"* | Vindt `Lidl Belgium - Antwerp` maar weet niet dat er geen SF account is → **onvolledig** | Graph ziet de ontbrekende link → **correct gap gedetecteerd** |
| *"Who is the contact behind my most expensive order?"* | Vindt `AB InBev - Leuven` (SS) maar matcht niet aan `Anheuser-Busch InBev` (SF) → **geen contact** | Graph kent de alias → **contact gevonden** |